# Korean Civil Act citation accuracy: base, LoRA and retrieval

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/No1Joon/oh-my-notebooks/blob/main/notebooks/legal_citation_lora_rag.ipynb)

Select a GPU runtime and run all cells. This controlled experiment predicts a case’s annotated statute number. The score concerns the historical reference field. Public [precedents](https://huggingface.co/datasets/woalsdnd/law.go.kr) (MIT on the dataset card), [Qwen2.5-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct) (Apache-2.0), and [Nanum Gothic](https://github.com/google/fonts/tree/main/ofl/nanumgothic) (OFL) are downloaded for this run. No account or upload is required. Tested runtime versions appear in the final report.

The cells below assemble one Python program in memory. The final cell runs it inside a disposable workspace. Set CHART_LANG to ko, en or both before execution (default ko).

In [ ]:
import json, os, subprocess, sys, tempfile
from pathlib import Path
from IPython.display import Image, display
program = []

## Sources and experiment settings

Pinned model and dataset revisions. Only the files needed for the run are fetched into a new temporary directory.

In [ ]:
program.append(r'''
import hashlib
import json
import os
import platform
import random
import re
import shutil
import subprocess
import sys
import tempfile
import unicodedata
import zipfile
from collections import Counter
from pathlib import Path
from urllib.request import urlopen

SEED = 20261125
TRAIN, TEST = 192, 40
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"  # Qwen, Apache-2.0
REV = "7ae557604adf67be50417f59c2c2f167def9a775"
CORPUS_REV = "972b50e77f89b2d85f5fa950030e44a50861003e"
CORPUS_URL = f"https://huggingface.co/datasets/woalsdnd/law.go.kr/resolve/{CORPUS_REV}/crawled_data.zip"
CORPUS_HASH = "7147a31cea7d0206da0cad0d830fbb6dd81b0c83bc539cf0c2f987789e798b2b"
WEIGHT_HASH = "fdf756fa7fcbe7404d5c60e26bff1a0c8b8aa1f72ced49e7dd0210fe288fb7fe"
FONT_URL = "https://raw.githubusercontent.com/google/fonts/main/ofl/nanumgothic/NanumGothic-Regular.ttf"


def download(url, path, digest=None):
    with urlopen(url, timeout=180) as source, path.open("wb") as target:
        shutil.copyfileobj(source, target)
    if digest:
        with path.open("rb") as handle:
            if hashlib.file_digest(handle, "sha256").hexdigest() != digest:
                raise ValueError(f"Checksum mismatch: {path.name}")
''')

## Split Korean precedent records; index only training issues.

Keep cases whose reference field lists one Civil Act article, optionally with a paragraph. Skip issues that already quote an Arabic-numbered article. Random ordering precedes the 192/40 split.

In [ ]:
program.append(r'''
def select_cases(archive):
    # Snapshot from Korean Ministry of Government Legislation, dataset card marked MIT.
    # https://huggingface.co/datasets/woalsdnd/law.go.kr
    names_seen, issues_seen = set(), set()
    cases = []
    reference = re.compile(r"민법\s*제\s*(\d+)\s*조(?:\s*제\s*\d+\s*항)?[.\s]*")
    with zipfile.ZipFile(archive) as corpus:
        names = sorted(n for n in corpus.namelist() if re.fullmatch(r"crawled_data/licPrec\d+\.txt", n))
        random.Random(SEED).shuffle(names)
        for filename in names:
            text = unicodedata.normalize("NFKC", corpus.read(filename).decode("utf8"))
            parts = re.split(r"【([^】\n]+)】", text)
            sections = dict(zip(parts[1::2], parts[2::2]))
            gold = reference.fullmatch(sections.get("참조조문", "").strip())
            question = re.sub(r"\[\d+\]", " ", sections.get("판시사항", ""))
            question = " ".join(question.split())
            if not gold or len(question) < 30 or re.search(r"\d+\s*(?:조|條)|\bArticle\s+\d+", question, re.I):
                continue
            question = question[:240].strip()
            issue_key = re.sub(r"[\W_]", "", question)
            docket = re.findall(r"(?<!\d)(?:\d{4}|\d{2})[가-힣]{1,4}\d+(?!\d)", parts[0])
            if issue_key in issues_seen or any(key in names_seen for key in docket):
                continue
            names_seen.update(docket)
            issues_seen.add(issue_key)
            cases.append((question, int(gold.group(1))))
            if len(cases) == TRAIN + TEST:
                break
    assert len(cases) == TRAIN + TEST
    return cases[:TRAIN], cases[TRAIN:]


def retrieve(train, test):
    from sklearn.feature_extraction.text import TfidfVectorizer
    import numpy as np

    index = TfidfVectorizer(analyzer="char", ngram_range=(2, 3))
    indexed = index.fit_transform([q for q, _ in train])
    scores = (index.transform([q for q, _ in test]) @ indexed.T).toarray()
    # Stable ties follow the seeded training order.
    return [[train[i] for i in np.argsort(-row, kind="stable")[:3]] for row in scores]
''')

## Score every recognized Korean article expression, including other or omitted acts.

A hit requires a Korean act name and article number. Unqualified Korean article expressions and outputs without a Korean article expression are counted separately. Paragraphs are ignored. Multiple references are deduplicated within a case.

In [ ]:
program.append(r'''
ARTICLE = re.compile(r"(?:제\s*)?(\d+)\s*조(?:\s*의\s*(\d+))?(?![가-힣]*원)")
# A statutory name immediately before the citation, allowing simple markup/particles.
LAW = re.compile(r"([가-힣]+(?:법률|법령|법|규칙|조례|시행령|시행규칙|헌장))\s*[」》”'’*]*\s*(?:의\s*)?$")
CARRY = re.compile(r"[\s,，、;및와과또는과~∼부터내지까지]*(?:제\s*\d+\s*(?:항|호)[\s,，、;]*)*")


def extract(answer):
    text = unicodedata.normalize("NFKC", answer)
    refs = set()
    last_end, previous = 0, None
    for match in ARTICLE.finditer(text):
        gap = text[last_end:match.start()]
        named = LAW.search(gap)
        if named:
            law = named.group(1)
        elif previous and CARRY.fullmatch(gap):
            law = previous
        else:
            law = "?"
        number = str(int(match.group(1)))
        if match.group(2):
            number += "-" + str(int(match.group(2)))
        refs.add((law, number))
        previous, last_end = law, match.end()
    return refs


def score(answer, number):
    refs = extract(answer)
    gold = ("민법", str(number))
    named = {r for r in refs if r[0] != "?"}
    return dict(hit=gold in refs, extra=bool(named - {gold}), no_article=not refs,
                exact=refs == {gold}, unnamed=any(law == "?" for law, _ in refs))


assert score("형법 제750조", 750)["extra"] and not score("형법 제750조", 750)["hit"]
assert score("민법 제750조 및 제751조", 750)["extra"]
assert score("민법 제750조, 형법 제9조", 750)["hit"]
assert score("민법 제750조 제1항", 750)["exact"] and not score("민법 제750조의2", 750)["hit"]
assert score("제750조", 750)["unnamed"] and score("근거 부족", 750)["no_article"]
''')

## Train an adapter on target tokens only and evaluate the same held-out issues.

Base and retrieval use identical frozen weights. LoRA sees only training questions and gold labels. The retrieval index contains only training issues and returns three with their labels. All 40 held-out prompts are decoded greedily.

In [ ]:
program.append(r'''
def run_model(work, train, test, neighbors):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    if not torch.cuda.is_available():
        raise RuntimeError("Select a Colab GPU runtime, then rerun.")
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    local = work / "model"
    local.mkdir()
    for filename in ("config.json", "generation_config.json", "tokenizer.json", "tokenizer_config.json",
                     "vocab.json", "merges.txt", "model.safetensors"):
        digest = WEIGHT_HASH if filename.endswith(".safetensors") else None
        print("Retrieving", filename, flush=True)
        download(f"https://huggingface.co/{MODEL}/resolve/{REV}/{filename}?download=true", local / filename, digest)
    tok = AutoTokenizer.from_pretrained(local, local_files_only=True)
    model = AutoModelForCausalLM.from_pretrained(local, dtype=torch.float16, local_files_only=True).to("cuda")
    system = ("당신은 대한민국 민법 판례의 참조조문을 식별합니다. 판시사항을 읽고 가장 관련된 조문을 "
              "'민법 제N조' 형식으로 쓰세요. 모르면 '모름'으로 답하세요. 설명은 쓰지 마세요.")

    def prompt(q, evidence=()):
        context = ""
        if evidence:
            context = "참고할 기존 판례:\n" + "\n".join(
                f"- {issue[:120]} (참조조문: 민법 제{article}조)" for issue, article in evidence) + "\n\n"
        messages = [{"role": "system", "content": system},
                    {"role": "user", "content": context + "판시사항: " + q}]
        return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    def evaluate(contexts):
        model.eval()
        counts, cutoffs = Counter(), 0
        tok.padding_side = "left"
        for start in range(0, len(test), 8):
            batch = test[start:start+8]
            prompts = [prompt(q, evidence) for (q, _), evidence in zip(batch, contexts[start:start+len(batch)])]
            inp = tok(prompts, padding=True, add_special_tokens=False, return_tensors="pt").to("cuda")
            with torch.inference_mode():
                out = model.generate(**inp, do_sample=False, num_beams=1, max_new_tokens=48,
                                     pad_token_id=tok.pad_token_id, use_cache=True)
            for j, (generated, (_, gold)) in enumerate(zip(out[:, inp.input_ids.shape[1]:], batch)):
                measured = score(tok.decode(generated, skip_special_tokens=True), gold)
                counts.update(measured)
                if contexts[start+j]:
                    covered = any(a == gold for _,a in contexts[start+j])
                    counts["hit_with_gold_in_context" if covered else "hit_without_gold_in_context"] += int(measured["hit"])
                cutoffs += int(tok.eos_token_id not in generated.tolist())
        counts["cap"] = cutoffs
        return dict(counts)

    empty = [()] * TEST
    results = {"base": evaluate(empty), "rag": evaluate(neighbors)}
    print("Evaluation of base and retrieval conditions complete.", flush=True)
    # The adapter computes W(x) + (alpha/r) B(A(x)). B starts at zero,
    # so inserting the adapter initially reproduces the frozen base layer.
    class LoRAProjection(torch.nn.Module):
        def __init__(self, original, rank=8, alpha=16):
            super().__init__()
            self.original = original
            self.scaling = alpha / rank
            self.down = torch.nn.Linear(original.in_features, rank, bias=False,
                                        device=original.weight.device, dtype=torch.float32)
            self.up = torch.nn.Linear(rank, original.out_features, bias=False,
                                      device=original.weight.device, dtype=torch.float32)
            torch.nn.init.zeros_(self.up.weight)

        def forward(self, x):
            base = self.original(x)
            update = self.up(self.down(x.float())) * self.scaling
            return base + update.to(base.dtype)

    model.requires_grad_(False)
    for layer in model.model.layers:
        for name in ("q_proj", "v_proj"):
            original = getattr(layer.self_attn, name)
            setattr(layer.self_attn, name, LoRAProjection(original))
    model.train()
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=2e-4,
                                  weight_decay=0.0)
    order = list(range(TRAIN))
    random.Random(SEED + 1).shuffle(order)
    loss_first, loss_last = None, None
    for i in range(0, TRAIN, 4):
        encoded, labels = [], []
        for ix in order[i:i+4]:
            q, article = train[ix]
            prefix = tok.encode(prompt(q), add_special_tokens=False)
            completion = tok.encode(f"민법 제{article}조", add_special_tokens=False) + [tok.eos_token_id]
            encoded.append(prefix + completion)
            labels.append([-100] * len(prefix) + completion)
        longest = max(map(len, encoded))
        ids = torch.tensor([v + [tok.pad_token_id] * (longest-len(v)) for v in encoded], device="cuda")
        mask = torch.tensor([[1]*len(v) + [0]*(longest-len(v)) for v in encoded], device="cuda")
        target = torch.tensor([v + [-100]*(longest-len(v)) for v in labels], device="cuda")
        optimizer.zero_grad(set_to_none=True)
        loss = model(input_ids=ids, attention_mask=mask, labels=target, use_cache=False).loss
        if not torch.isfinite(loss):
            raise ValueError("Non-finite training loss")
        loss.backward()
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
        optimizer.step()
        if i == 0:
            loss_first = loss.item()
        loss_last = loss.item()
    results["lora"] = evaluate(empty)
    return dict(metrics=results, train=TRAIN, test=TEST,
                trainable=sum(p.numel() for p in model.parameters() if p.requires_grad),
                train_articles=len(set(a for _, a in train)), test_articles=len(set(a for _, a in test)),
                train_coverage=sum(any(a == g for _, a in train) for _,g in test),
                top3_coverage=sum(any(a == g for _,a in docs) for docs,(_,g) in zip(neighbors,test)),
                top1_hits=sum(docs[0][1] == g for docs, (_,g) in zip(neighbors,test)),
                first_loss=round(loss_first,4), last_loss=round(loss_last,4),
                gpu=torch.cuda.get_device_name(0), python=platform.python_version(),
                versions={name: __import__(name).__version__ for name in ("torch","transformers","sklearn","matplotlib")})
''')

## Render charts and manage a disposable runtime workspace.

A child process isolates model and optimizer memory, cache locations and generated bytecode. Source data and model weights are removed when the child returns, including on failure. Only aggregate counters and plots are shown.

In [ ]:
program.append(r'''
def plot(result, work, output, language):
    import matplotlib.pyplot as plt
    from matplotlib import font_manager

    if language not in ("ko","en","both"):
        raise ValueError("CHART_LANG must be ko, en or both")
    languages = ["ko","en"] if language == "both" else [language]
    if "ko" in languages:
        font = work / "NanumGothic-Regular.ttf"  # Google Fonts, SIL Open Font License 1.1.
        download(FONT_URL, font)
        font_manager.fontManager.addfont(str(font))
        plt.rcParams["font.family"] = font_manager.FontProperties(fname=str(font)).get_name()
    for lang in languages:
        ko = lang == "ko"
        keys = ("hit","extra","no_article","exact")
        labels = ["정답 포함","추가 명시 인용","조 번호 없음","정확히 일치"] if ko else [
            "Any gold hit","Extra named citation","No article number","Exact match"]
        fig, ax = plt.subplots(figsize=(9,4.5),dpi=170,layout="constrained")
        for j,(method,color) in enumerate(zip(("base","lora","rag"),("#818589","#3e718c","#ad623d"))):
            bars = ax.bar([i + .25*(j-1) for i in range(4)],
                          [result["metrics"][method][k] for k in keys], .24,
                          label={"base":"기본 모델" if ko else "Base","lora":"LoRA","rag":"RAG"}[method],
                          color=color)
            ax.bar_label(bars,padding=2,fontsize=9)
        ax.set_xticks(range(4),labels)
        ax.set_ylabel("사건 수" if ko else "Cases")
        ax.set_title(f"학습에서 제외한 민법 판례 {TEST}건" if ko else f"{TEST} held-out Korean Civil Act precedents")
        ax.set_ylim(0,TEST+8)
        ax.grid(axis="y",alpha=.15)
        ax.set_axisbelow(True)
        ax.legend(frameon=False,ncols=3)
        fig.savefig(output / f"citation-{lang}.png")
        plt.close(fig)


def worker(work, output):
    archive = work / "precedents.zip"
    download(CORPUS_URL, archive, CORPUS_HASH)
    train, test = select_cases(archive)
    archive.unlink()
    neighbors = retrieve(train, test)
    print("Prepared held-out evaluation split.", flush=True)
    result = run_model(work, train, test, neighbors)
    plot(result, work, output, os.getenv("CHART_LANG","ko"))
    (output/"metrics.json").write_text(json.dumps(result, indent=2), encoding="utf8")


def launch(output):
    output.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix="legal-citation-") as directory:
        work = Path(directory)
        # The child exits before deletion. Library caches and compiler artifacts
        # are directed to this run even if the notebook kernel imported HF before.
        env = os.environ.copy()
        for key in ("HF_HOME","HF_HUB_CACHE","HF_XET_CACHE","HF_DATASETS_CACHE","TORCH_HOME",
                    "XDG_CACHE_HOME","MPLCONFIGDIR","TORCHINDUCTOR_CACHE_DIR","TRITON_CACHE_DIR",
                    "CUDA_CACHE_PATH","TMPDIR","NUMBA_CACHE_DIR"):
            cache = work / key.lower()
            cache.mkdir()
            env[key] = str(cache)
        env.update(HF_HUB_OFFLINE="1", TOKENIZERS_PARALLELISM="false", MPLBACKEND="Agg", PYTHONDONTWRITEBYTECODE="1", HF_HUB_DISABLE_PROGRESS_BARS="1")
        proc = subprocess.run([sys.executable, str(Path(__file__).resolve()), "--worker",
                               str(work), str(output.resolve())], cwd=work, env=env)
    assert not work.exists(), "Disposable model and dataset workspace still exists"
    print("Verified cleanup of the model, adapter workspace, corpus and per-run caches.", flush=True)
    proc.check_returncode()


if __name__ == "__main__":
    if len(sys.argv) == 4 and sys.argv[1] == "--worker":
        worker(Path(sys.argv[2]), Path(sys.argv[3]))
    elif len(sys.argv) == 2:
        launch(Path(sys.argv[1]))
    else:
        raise SystemExit("Usage: python experiment.py OUTPUT_DIRECTORY")
''')

## Run, display aggregate results and clean up

The intermediate program, generated PNG files and metrics file also live in a temporary directory. Notebook outputs contain only aggregate measurements and the requested charts.

In [ ]:
with tempfile.TemporaryDirectory(prefix="citation-notebook-") as output_dir:
    output_dir = Path(output_dir)
    script = output_dir / "experiment.py"
    script.write_text("\n".join(program), encoding="utf8")
    with subprocess.Popen([sys.executable, str(script), str(output_dir)], stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True) as child:
        for line in child.stdout:
            print(line.rstrip(), flush=True)
        if child.wait():
            raise RuntimeError("Experiment failed; inspect the messages above.")
    report = json.loads((output_dir / "metrics.json").read_text())
    print("RESULT", json.dumps(report, ensure_ascii=False))
    for language in ("ko", "en"):
        chart = output_dir / f"citation-{language}.png"
        if chart.exists():
            display(Image(data=chart.read_bytes()))
assert not output_dir.exists()
print("Verified removal of notebook staging files.")